<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/Mem0_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q mem0ai langgraph langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.7/325.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.9/370.9 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 38.2 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["MEM0_API_KEY"] = "m0-u4WNGXhUGVKhds7MXqurq8IMPDN5TFUgNPyfeulZ"

In [ ]:
from mem0 import MemoryClient

memory = MemoryClient(api_key=os.environ["MEM0_API_KEY"])
print(" Mem0 Platform ready!")


memory.add(
    messages=[{"role": "user", "content": "test connection"}],
    user_id="test_001"
)
print(" Add works!")

results = memory.search(
    query="test",
    filters={"user_id": "test_001"}
)
print(" Search works!")
print("Results:", results)

 Mem0 Platform ready!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


 Add works!
 Search works!
Results: {'results': []}


In [ ]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
import operator

class AgentState(TypedDict):
    messages : Annotated[List[BaseMessage], operator.add]
    user_id  : str
    memory_context: str

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)
print("LLM ready!")

LLM ready!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from langchain_core.messages import SystemMessage

def retrieve_memory(state: AgentState) -> AgentState:
    user_id      = state["user_id"]
    last_message = state["messages"][-1].content

    results = memory.search(
        query=last_message,
        filters={"user_id": user_id},
        limit=5
    )


    hits = results if isinstance(results, list) else results.get("results", [])

    memory_text = "\n".join(
        f"- {r['memory']}" for r in hits
    ) or "No relevant memories found."

    print(f"\n[Memory Retrieved]\n{memory_text}")
    return {**state, "memory_context": memory_text}



def generate_response(state: AgentState) -> AgentState:
    memory_context = state["memory_context"]
    messages       = state["messages"]

    system_prompt = f"""You are a helpful assistant with memory.
Use the following relevant memories to personalize your response:

{memory_context}

Always be concise and reference past context when relevant."""

    full_messages = [SystemMessage(content=system_prompt)] + messages
    response      = llm.invoke(full_messages)

    print(f"\n[Agent Response]\n{response.content}")
    return {**state, "messages": [AIMessage(content=response.content)]}


def save_memory(state: AgentState) -> AgentState:
    user_id  = state["user_id"]
    messages = state["messages"]

    formatted = [
        {
            "role":    "user"      if isinstance(m, HumanMessage) else "assistant",
            "content": m.content
        }
        for m in messages[-2:]
    ]

    memory.add(
        messages=formatted,
        user_id=user_id
    )
    print("\n[Memory Saved ")
    return state

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

graph.add_node("retrieve_memory",   retrieve_memory)
graph.add_node("generate_response", generate_response)
graph.add_node("save_memory",       save_memory)

graph.set_entry_point("retrieve_memory")
graph.add_edge("retrieve_memory",   "generate_response")
graph.add_edge("generate_response", "save_memory")
graph.add_edge("save_memory",       END)

agent = graph.compile()
print("LangGraph agent compiled!")

LangGraph agent compiled!


In [ ]:
def chat(user_message: str, user_id: str = "user_001"):
    print(f"\n{'='*50}")
    print(f"User: {user_message}")
    print('='*50)

    initial_state = AgentState(
        messages      =[HumanMessage(content=user_message)],
        user_id       =user_id,
        memory_context=""
    )

    result = agent.invoke(initial_state)
    return result["messages"][-1].content

In [ ]:
dummy = [

    ("My name is Alex, I am a ML engineer at a healthcare startup in London", "user_001"),
    ("I use Python, LangChain and LlamaIndex for building RAG pipelines",     "user_001"),
    ("I am building a medical document Q&A system using GPT-4 and Pinecone",  "user_001"),
    ("I know Sarah, she is a data scientist, we met at a conference",          "user_001"),


    ("My name is Sarah, I am a data scientist at a fintech company in New York", "user_002"),
    ("I use PyTorch and HuggingFace for NLP tasks",                              "user_002"),
    ("I am building a fraud detection model using graph neural networks",         "user_002"),

    ("My name is James, I am a DevOps engineer learning AI on the side",    "user_003"),
    ("I use Kubernetes and Docker for infrastructure",                        "user_003"),
    ("I am experimenting with LangGraph and CrewAI for multi agent systems", "user_003"),
]

print("Seeding memories...\n")
for message, user_id in dummy:
    memory.add(
        messages=[{"role": "user", "content": message}],
        user_id=user_id
    )
    print(f"[{user_id}]  {message[:60]}...")

print("\n All seeded!")

Seeding memories...

[user_001]  My name is Alex, I am a ML engineer at a healthcare startup ...
[user_001]  I use Python, LangChain and LlamaIndex for building RAG pipe...
[user_001]  I am building a medical document Q&A system using GPT-4 and ...
[user_001]  I know Sarah, she is a data scientist, we met at a conferenc...
[user_002]  My name is Sarah, I am a data scientist at a fintech company...
[user_002]  I use PyTorch and HuggingFace for NLP tasks...
[user_002]  I am building a fraud detection model using graph neural net...
[user_003]  My name is James, I am a DevOps engineer learning AI on the ...
[user_003]  I use Kubernetes and Docker for infrastructure...
[user_003]  I am experimenting with LangGraph and CrewAI for multi agent...

 All seeded!


In [ ]:
for user_id in ["user_001", "user_002", "user_003"]:
    print(f"\n{'='*40}")
    print(f"Memories for {user_id}")
    print('='*40)

    results = memory.search(
        query="who is this person and what do they work on?",
        filters={"user_id": user_id},
        limit=10
    )
    hits = results if isinstance(results, list) else results.get("results", [])
    for r in hits:
        print(f"  - {r['memory']}")


Memories for user_001
  - User's name is Alex and works as a machine learning engineer at a healthcare startup based in London.
  - User is building a medical document question-and-answer system using GPT-4 and Pinecone
  - User uses Python, LangChain, and LlamaIndex to build Retrieval-Augmented Generation (RAG) pipelines

Memories for user_002
  - User's name is Sarah and she works as a data scientist at a fintech company in New York
  - User uses PyTorch and HuggingFace for natural language processing (NLP) tasks

Memories for user_003
  - User's name is James and he works as a DevOps engineer while learning AI on the side
  - User is experimenting with LangGraph and CrewAI for multi-agent systems
  - User uses Kubernetes and Docker for infrastructure


In [ ]:

chat("Hi! My name is Alex and I love building AI agents.", user_id="user_001")
chat("What kind of projects do you think I'd enjoy?",      user_id="user_001")
chat("Can you remind me what I told you about myself?",    user_id="user_001")


User: Hi! My name is Alex and I love building AI agents.

[Memory Retrieved]
- User's name is Alex and works as a machine learning engineer at a healthcare startup based in London.
- User uses Python, LangChain, and LlamaIndex to build Retrieval-Augmented Generation (RAG) pipelines
- User knows Sarah, who works as a data scientist, and they met at a conference
- User is building a medical document question-and-answer system using GPT-4 and Pinecone


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



[Agent Response]
Hi Alex! It's great to hear from you again. I remember you're working on a medical document question-and-answer system using GPT-4 and Pinecone. How's that project going? Are you integrating any new features with Python, LangChain, or LlamaIndex?

[Memory Saved 

User: What kind of projects do you think I'd enjoy?

[Memory Retrieved]
- User knows Sarah, who works as a data scientist, and they met at a conference
- User's name is Alex and works as a machine learning engineer at a healthcare startup based in London.
- User is building a medical document question-and-answer system using GPT-4 and Pinecone
- User uses Python, LangChain, and LlamaIndex to build Retrieval-Augmented Generation (RAG) pipelines


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



[Agent Response]
Given your background as a machine learning engineer at a healthcare startup and your current work on a medical document question-and-answer system using GPT-4 and Pinecone, you might enjoy projects that involve:

1. **Healthcare AI Innovations**: Building predictive models for patient outcomes or personalized treatment plans using machine learning.

2. **Advanced NLP Applications**: Expanding your current project to include more complex natural language processing tasks, such as summarization or sentiment analysis in medical documents.

3. **RAG Pipeline Enhancements**: Exploring new ways to optimize Retrieval-Augmented Generation pipelines using Python, LangChain, and LlamaIndex, perhaps by integrating more diverse data sources or improving retrieval accuracy.

4. **Collaborative Projects with Data Scientists**: Since you know Sarah, who works as a data scientist, you might enjoy collaborating on interdisciplinary projects that combine data science and machine learn

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



[Agent Response]
Certainly, Alex! You mentioned that you're a machine learning engineer at a healthcare startup based in London. You're currently working on building a medical document question-and-answer system using GPT-4 and Pinecone. You use Python, LangChain, and LlamaIndex to build Retrieval-Augmented Generation (RAG) pipelines. Additionally, you know Sarah, who works as a data scientist, and you met her at a conference. Let me know if there's anything else you'd like to add or update!

[Memory Saved 


"Certainly, Alex! You mentioned that you're a machine learning engineer at a healthcare startup based in London. You're currently working on building a medical document question-and-answer system using GPT-4 and Pinecone. You use Python, LangChain, and LlamaIndex to build Retrieval-Augmented Generation (RAG) pipelines. Additionally, you know Sarah, who works as a data scientist, and you met her at a conference. Let me know if there's anything else you'd like to add or update!"

In [ ]:

def chat_with_debug(user_message: str, user_id: str = "user_001"):
    print(f"\n{'='*50}")
    print(f"USER ({user_id}): {user_message}")
    print('='*50)

    hits = memory.search(
        query=user_message,
        filters={"user_id": user_id},
        limit=5
    )
    raw = hits if isinstance(hits, list) else hits.get("results", [])

    print("\n[What Mem0 found in graph]")
    if raw:
        for r in raw:
            print(f"  score: {r.get('score', 'N/A'):.3f} | {r['memory']}")
    else:
        print("  (nothing found yet)")

    response = chat(user_message, user_id=user_id)
    return response

chat_with_debug("What does Alex work on?",            user_id="user_001")
chat_with_debug("What frameworks does Sarah use?",    user_id="user_002")
chat_with_debug("What is James experimenting with?",  user_id="user_003")


USER (user_001): What does Alex work on?

[What Mem0 found in graph]
  score: 0.301 | User's name is Alex and works as a machine learning engineer at a healthcare startup based in London.
  score: 0.257 | Assistant suggested Alex could work on healthcare AI innovations, such as building predictive models for patient outcomes or personalized treatment plans, leveraging his machine learning expertise at a healthcare startup.
  score: 0.232 | Assistant advised developing AI ethics frameworks for healthcare, focusing on fairness, transparency, and patient privacy, aligning with Alex's role in a healthcare startup.
  score: 0.225 | Assistant proposed enhancing Retrieval-Augmented Generation pipelines by integrating more diverse data sources and improving retrieval accuracy, using Alex's skills with Python, LangChain, and LlamaIndex.
  score: 0.192 | Assistant recommended expanding the current medical document Q&A system to advanced NLP tasks like summarization or sentiment analysis of medi

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



[Agent Response]
Alex works as a machine learning engineer at a healthcare startup in London. He is currently building a medical document question-and-answer system using GPT-4 and Pinecone. Additionally, Alex uses Python, LangChain, and LlamaIndex to develop Retrieval-Augmented Generation (RAG) pipelines. He is passionate about building AI agents and frequently explores new techniques for creating intelligent autonomous systems.

[Memory Saved 

USER (user_002): What frameworks does Sarah use?

[What Mem0 found in graph]
  score: 0.301 | User's name is Sarah and she works as a data scientist at a fintech company in New York
  score: 0.203 | User uses PyTorch and HuggingFace for natural language processing (NLP) tasks
  score: 0.163 | User is building a fraud detection model using graph neural networks

User: What frameworks does Sarah use?

[Memory Retrieved]
- User's name is Sarah and she works as a data scientist at a fintech company in New York
- User uses PyTorch and HuggingFace 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



[Agent Response]
You use PyTorch and HuggingFace for your natural language processing tasks.

[Memory Saved 

USER (user_003): What is James experimenting with?

[What Mem0 found in graph]
  score: 0.432 | User is experimenting with LangGraph and CrewAI for multi-agent systems
  score: 0.261 | User's name is James and he works as a DevOps engineer while learning AI on the side
  score: 0.153 | User uses Kubernetes and Docker for infrastructure

User: What is James experimenting with?

[Memory Retrieved]
- User is experimenting with LangGraph and CrewAI for multi-agent systems
- User's name is James and he works as a DevOps engineer while learning AI on the side
- User uses Kubernetes and Docker for infrastructure


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



[Agent Response]
James is experimenting with LangGraph and CrewAI for multi-agent systems.

[Memory Saved 


'James is experimenting with LangGraph and CrewAI for multi-agent systems.'